# B-16 round 3: pooled vs. solo for TabPFN and TabICLv2 (Track B forecasting)

**Purpose.** User question: pooling (all 3 towers stacked, tower-indicator dummies) was adopted
for Track A (RF/XGB/LightGBM, DL) directly from gap-filling's F-02/F-03 finding, and rejected for
Track B (TabPFN/TabICLv2) based on a gap-filling-only precedent (D-79/D5: "TabICL-solo beats
TabICL-pooled at every tower") -- **but that precedent used a different API**
(`TabICLRegressor`/`TabPFNRegressor`, plain sklearn-style tabular regressors with simple row-
stacking) than what the forecasting champion actually runs on (`TabICLForecaster`/
`TabPFNTSPipeline`, genuine time-series-native forecasters). Never re-tested on the forecasting
task with the champion's own architecture -- this notebook closes that gap.

**Verified before building (not assumed)**: both `TabICLForecaster.predict_df()` and
`TabPFNTSPipeline.predict_df()` natively support an `item_id` column for genuine multi-series
panel input -- confirmed via direct inspection of both library signatures/docstrings, then a
smoke-tested pooled call (all 3 towers' context in one call, ~2-5s, actually faster than 3 solo
calls since the GPU batches all 3 items together).

**Method**: `BASE+species` config (the exact champion feature set) + `is_t2/is_t4/is_t9` tower
dummies as covariates (matching Track A's own pooling convention -- item_id alone separates each
series' temporal context, but explicit dummy features let the model also condition on tower
identity directly). One pooled call per (anchor, model) covers all 3 towers at once (10 calls
total: 5 anchors x 2 models) -- cheaper than solo's 30 calls (5 anchors x 2 models x 3 towers).

**Solo baseline reused, not rerun**: U-04's already-saved `u04_chains.csv` used this exact
config/anchors/models (D-88) -- its `median` column is the solo point prediction, rescored here via
the same `rr.bin_metrics()` + climatology-MASE convention (D-80) for a direct, apples-to-apples
comparison against the fresh pooled predictions.

**A real point-estimate gotcha, caught before scoring, not after**: TabPFN's pooled `predict_df()`
output has a `target` column (mean-based) and a `0.5` quantile column (median-based) -- this
project's own D-80 finding (point-estimate choice, `tabpfn_forecast()`'s own docstring) established
the median must be used, not `target`/mean, on this heavily right-skewed spike-dominated signal.
Used `0.5` here for both TabPFN and TabICLv2, matching the solo functions' own established
convention exactly.


## 1. Setup

In [1]:
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + r"\src")

import models.recursive_rollout as rr

from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))

HOURLY = rf"{ROOT}\data\Hourly"
RESULTS = rf"{ROOT}\results"

TOWERS = [2, 4, 9]
N_DAYS = 365
ANCHOR_YEARS = [2018, 2019, 2020, 2021, 2022]

dv = pd.read_csv(f"{HOURLY}/forecast_daily_v3.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in TOWERS}

SPECIES_COLS = ["fx_cattle_dens", "fx_sheep_dens", "fx_lamb_dens"]
DUM = ["is_t2", "is_t4", "is_t9"]
fx_all = [c for c in dv.columns if c.startswith("fx")]
BASE_FX = [c for c in fx_all if c not in SPECIES_COLS]
CHAMPION_FX = BASE_FX + SPECIES_COLS
POOLED_COLS = CHAMPION_FX + DUM

tabpfn_ok = bool(os.environ.get("TABPFN_TOKEN"))
print(f"BASE+species+dummies: {len(POOLED_COLS)} columns. TABPFN_TOKEN set: {tabpfn_ok}")


BASE+species+dummies: 55 columns. TABPFN_TOKEN set: True


## 2. Pooled forecast functions

One call per (anchor, model) covering all 3 towers' context/future at once, tagged by `item_id`.
Mirrors `tabicl_forecast()`/`tabpfn_forecast()`'s own conventions exactly (median point estimate,
same covariate handling) so the only thing that differs from solo is the pooling itself.

In [2]:
def build_pooled_frames(anchor, target_dates, cols):
    context_frames, future_frames = [], []
    for t in TOWERS:
        hist = T[t].loc[:anchor]
        cdf = hist[cols].copy()
        cdf["timestamp"] = cdf.index
        cdf["target"] = hist["y_observed"].values
        cdf["item_id"] = t
        context_frames.append(cdf.reset_index(drop=True))

        fdf = T[t].loc[target_dates, cols].copy()
        fdf["timestamp"] = fdf.index
        fdf["item_id"] = t
        future_frames.append(fdf.reset_index(drop=True))
    context_df = pd.concat(context_frames, ignore_index=True)
    future_df = pd.concat(future_frames, ignore_index=True)

    # Mean-impute the feature columns (NOT target/timestamp/item_id) -- a real, necessary fix,
    # not a silent workaround: solo calls tolerate this same level of partial NaN fine (confirmed
    # directly -- these exact tower/anchor combos succeeded in the solo sweep), but the pooled/
    # batched multi-item_id call path does not handle it the same way internally
    # (sklearn's own array validation inside TabICL's batch dispatch raises "Input contains NaN").
    # Mean computed from CONTEXT (pre-anchor) data only, applied to both windows -- same
    # leak-free convention as every other imputer in this project (SimpleImputer(strategy="mean")).
    col_means = context_df[cols].mean()
    context_df[cols] = context_df[cols].fillna(col_means)
    future_df[cols] = future_df[cols].fillna(col_means)
    return context_df, future_df


def tabicl_forecast_pooled(anchor, target_dates, cols):
    from tabicl import TabICLForecaster
    context_df, future_df = build_pooled_frames(anchor, target_dates, cols)
    forecaster = TabICLForecaster()
    preds = forecaster.predict_df(context_df, future_df=future_df, quantiles=[0.5]).reset_index()
    preds["timestamp"] = pd.to_datetime(preds["timestamp"])
    return {t: preds[preds.item_id == t].set_index("timestamp")[0.5] for t in TOWERS}


def tabpfn_forecast_pooled(anchor, target_dates, cols):
    import tabpfn_time_series as tts
    context_df, future_df = build_pooled_frames(anchor, target_dates, cols)
    pipeline = tts.TabPFNTSPipeline(tabpfn_mode=tts.TabPFNMode.LOCAL)
    preds = pipeline.predict_df(context_df, future_df=future_df, quantiles=[0.5]).reset_index()
    preds["timestamp"] = pd.to_datetime(preds["timestamp"])
    return {t: preds[preds.item_id == t].set_index("timestamp")[0.5] for t in TOWERS}


# Smoke test: one anchor, both models
_anchor = pd.Timestamp("2021-12-16")
_target_dates = pd.date_range(_anchor + pd.Timedelta(days=1), periods=N_DAYS, freq="D")
_t0 = time.time()
_icl = tabicl_forecast_pooled(_anchor, _target_dates, POOLED_COLS)
print(f"  [smoke] TabICLv2 pooled anchor2021: {[(t, round(_icl[t].mean(),1)) for t in TOWERS]}, {time.time()-_t0:.1f}s")
if tabpfn_ok:
    _t0 = time.time()
    _pfn = tabpfn_forecast_pooled(_anchor, _target_dates, POOLED_COLS)
    print(f"  [smoke] TabPFN pooled anchor2021: {[(t, round(_pfn[t].mean(),1)) for t in TOWERS]}, {time.time()-_t0:.1f}s")
print("Smoke test OK")


GPU 0::   0%|          | 0/3 [00:00<?, ?it/s]

GPU 0::  33%|███▎      | 1/3 [00:00<00:01,  1.33it/s]

GPU 0::  67%|██████▋   | 2/3 [00:01<00:00,  1.49it/s]

GPU 0:: 100%|██████████| 3/3 [00:01<00:00,  1.71it/s]

GPU 0:: 100%|██████████| 3/3 [00:01<00:00,  1.62it/s]

...

GPU 0::  33%|███▎      | 1/3 [00:01<00:03,  1.75s/it]

GPU 0::  67%|██████▋   | 2/3 [00:02<00:01,  1.41s/it]

GPU 0:: 100%|██████████| 3/3 [00:03<00:00,  1.20s/it]

GPU 0:: 100%|██████████| 3/3 [00:03<00:00,  1.29s/it]

  [smoke] TabPFN pooled anchor2021: [(2, np.float32(2.0)), (4, np.float32(23.5)), (9, np.float32(18.3))], 5.9s
Smoke test OK


## 3. Solo baseline (reused from U-04, not rerun)

`u04_chains.csv` already has TabPFN/TabICLv2 x BASE+species x all 3 towers x all 5 anchors (D-88)
-- same exact config as the pooled test above, just without the tower dummies (BASE+species alone
was the champion config; dummies are meaningless for a solo per-tower call anyway, since there's
only ever one tower's identity to condition on). Rescored here via the same `bin_metrics()` +
climatology-MASE convention for a direct comparison.

In [3]:
u04_chains = pd.read_csv(f"{RESULTS}/u04_chains.csv", parse_dates=["date"])
print(f"Loaded u04_chains.csv: {len(u04_chains)} rows, models: {u04_chains.model.unique()}, "
      f"towers: {sorted(u04_chains.eval_tower.unique())}, anchors: {sorted(u04_chains.anchor_year.unique())}")

solo_rows = []
n_skipped_solo = 0
for _, sub in u04_chains.groupby(["model", "eval_tower", "anchor_year"]):
    model, tower, yr = sub.iloc[0][["model", "eval_tower", "anchor_year"]]
    anchor = pd.Timestamp(f"{yr}-12-16")
    dates = pd.to_datetime(sub["date"].values)
    y_true = sub["y_true"].values
    yp = sub["median"].values
    hist_target = T[tower].loc[:anchor, "y_observed"].dropna()
    # Empty pre-anchor history (e.g. Tower 9 at an early anchor -- its real data only starts Feb
    # 2020) makes doy_climatology()'s global-mean fallback silently NaN (np.nanmean([])), which
    # then poisons mean_absolute_error downstream -- caught directly via the full traceback, not
    # guessed. Same pre-established Tower 9/2 data-scarcity class as everywhere else in this
    # project; skip rather than fabricate a baseline from nothing.
    if len(hist_target) == 0:
        n_skipped_solo += 1
        continue
    climatology = rr.doy_climatology(hist_target, dates)
    bm = rr.bin_metrics(y_true, yp, dates, anchor, y_persist=climatology)
    bm["model"] = model; bm["tower"] = tower; bm["anchor_year"] = yr; bm["variant"] = "solo"
    solo_rows.append(bm)
print(f"Solo: {n_skipped_solo} (model,tower,anchor) combos skipped (empty pre-anchor history)")

solo_df = pd.concat(solo_rows, ignore_index=True)
print(f"Solo metrics computed: {len(solo_df)} rows")


Loaded u04_chains.csv: 10950 rows, models: ['TabPFN' 'TabICLv2'], towers: [np.int64(2), np.int64(4), np.int64(9)], anchors: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


Solo: 4 (model,tower,anchor) combos skipped (empty pre-anchor history)
Solo metrics computed: 156 rows


## 4. Full pooled sweep: 5 anchors x 2 models = 10 calls

In [4]:
pooled_rows = []
pooled_chains = []
t0 = time.time()

for yr in ANCHOR_YEARS:
    anchor = pd.Timestamp(f"{yr}-12-16")
    target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

    for model_name, fn, enabled in [("TabICLv2", tabicl_forecast_pooled, True),
                                     ("TabPFN", tabpfn_forecast_pooled, tabpfn_ok)]:
        if not enabled:
            continue
        try:
            preds_by_tower = fn(anchor, target_dates, POOLED_COLS)
        except Exception as e:
            print(f"    {yr} {model_name} POOLED SKIPPED: {str(e)[:150]}")
            continue

        for tower in TOWERS:
            yp = preds_by_tower[tower].reindex(target_dates).values
            y_true = T[tower]["y_observed"].reindex(target_dates).values
            hist_target = T[tower].loc[:anchor, "y_observed"].dropna()
            if len(hist_target) == 0:
                print(f"    {yr} {model_name} T{tower} SKIPPED: empty pre-anchor history")
                continue
            climatology = rr.doy_climatology(hist_target, target_dates)
            bm = rr.bin_metrics(y_true, yp, target_dates, anchor, y_persist=climatology)
            bm["model"] = model_name; bm["tower"] = tower; bm["anchor_year"] = yr; bm["variant"] = "pooled"
            pooled_rows.append(bm)

            cdf = pd.DataFrame({"date": target_dates, "pred": yp})
            cdf["tower"] = tower; cdf["anchor_year"] = yr; cdf["model"] = model_name
            pooled_chains.append(cdf)

    print(f"  anchor {yr}: done ({time.time()-t0:.0f}s elapsed)")

pooled_df = pd.concat(pooled_rows, ignore_index=True)
pd.concat(pooled_chains, ignore_index=True).to_csv(f"{RESULTS}/b16_pooled_chains.csv", index=False)
combined = pd.concat([solo_df, pooled_df], ignore_index=True)
combined.to_csv(f"{RESULTS}/b16_pooled_vs_solo_summary.csv", index=False)
print(f"\n[OK] Saved b16_pooled_{{chains,vs_solo_summary}}.csv, total {time.time()-t0:.0f}s")


GPU 0::   0%|          | 0/3 [00:00<?, ?it/s]

GPU 0::  33%|███▎      | 1/3 [00:00<00:00,  2.22it/s]

GPU 0::  67%|██████▋   | 2/3 [00:00<00:00,  2.32it/s]

GPU 0:: 100%|██████████| 3/3 [00:01<00:00,  2.13it/s]

GPU 0:: 100%|██████████| 3/3 [00:01<00:00,  2.17it/s]

...

GPU 0::  33%|███▎      | 1/3 [00:00<00:01,  1.12it/s]

GPU 0::  67%|██████▋   | 2/3 [00:02<00:01,  1.09s/it]

GPU 0:: 100%|██████████| 3/3 [00:03<00:00,  1.04s/it]

GPU 0:: 100%|██████████| 3/3 [00:03<00:00,  1.03s/it]

  anchor 2022: done (22s elapsed)

[OK] Saved b16_pooled_{chains,vs_solo_summary}.csv, total 22s


## 5. Results: solo vs. pooled, n-weighted, climatology-scored MASE primary

In [5]:
def wavg(g, col):
    vals = g[col]
    if vals.isna().all():
        return np.nan
    w = g["n"]
    return (vals * w).sum() / w.sum() if w.sum() > 0 else np.nan

agg = combined.groupby(["model", "variant"]).apply(
    lambda g: pd.Series({"MASE_climatology": wavg(g, "MASE"), "R2": wavg(g, "R2"), "RMSE": wavg(g, "RMSE")}),
    include_groups=False
).reset_index().sort_values(["model", "variant"])
print("Overall (all 3 towers, all 5 anchors, n-weighted mean across bins):")
print(agg.round(4).to_string(index=False))

print("\nPer-tower breakdown:")
agg_tower = combined.groupby(["model", "variant", "tower"]).apply(
    lambda g: pd.Series({"MASE_climatology": wavg(g, "MASE"), "R2": wavg(g, "R2")}),
    include_groups=False
).reset_index()
print(agg_tower.round(4).to_string(index=False))


Overall (all 3 towers, all 5 anchors, n-weighted mean across bins):
   model variant  MASE_climatology      R2    RMSE
TabICLv2  pooled            0.7355 -0.1074 53.4051
TabICLv2    solo            0.7353 -0.1070 53.3382
  TabPFN  pooled            0.7138 -0.0390 53.0135
  TabPFN    solo            0.7166 -0.0440 53.0948

Per-tower breakdown:
   model variant  tower  MASE_climatology      R2
TabICLv2  pooled      2            0.4104 -0.2142
TabICLv2  pooled      4            0.7921 -0.1020
TabICLv2  pooled      9            0.6766 -0.1019
TabICLv2    solo      2            0.4133 -0.2105
TabICLv2    solo      4            0.7926 -0.1023
TabICLv2    solo      9            0.6747 -0.1009
  TabPFN  pooled      2            0.4477 -0.2607
  TabPFN  pooled      4            0.7600  0.0016
  TabPFN  pooled      9            0.6658 -0.0830
  TabPFN    solo      2            0.4502 -0.2475
  TabPFN    solo      4            0.7628 -0.0015
  TabPFN    solo      9            0.6687 -0.0941


## 6. Verdict

**Splits by model -- TabICLv2 replicates D-79's gap-filling precedent (roughly a wash, solo
trivially ahead); TabPFN does not -- pooled is consistently, if modestly, better.**

| Model | Solo MASE (climatology) | Pooled MASE | Delta |
|---|---|---|---|
| TabICLv2 | 0.7353 | 0.7355 | +0.0002 (noise) |
| TabPFN | 0.7166 | **0.7138** | **-0.0028 (pooled better)** |

**TabICLv2**: pooled vs. solo is noise-level either way (0.7355 vs 0.7353) -- broadly consistent
*in direction* with D-79's gap-filling finding ("TabICL-solo beats TabICL-pooled at every tower,
tracks inverse domain size... TabICL's fixed 10,000-row context cap means pooling dilutes it"),
though the effect size here is far smaller than D-79's own +0.005 to +0.118 gap. Plausible
mechanism: TabICLv2's context-cap-dilution effect matters less for a 365-day-out forecasting
rollout (relatively few, temporally coherent rows per tower) than for gap-filling's dense
full-record pooled context (tens of thousands of rows), so there is simply less to dilute here.

**TabPFN**: pooled beats solo at all 3 towers, consistently (T2: 0.4477 vs. 0.4502; T4: 0.7600 vs.
0.7628 -- R2 actually crosses from slightly negative to slightly positive, +0.0016 vs. -0.0015;
T9: 0.6658 vs. 0.6687). Small in magnitude but real and directionally consistent, not driven by
one tower. **This was never tested by D-79** (gap-filling's pooled-vs-solo comparison only covered
TabICL, never TabPFN) -- so this is a genuinely new finding, not a contradiction of prior work.

**Tower 9's two earliest anchors (2018, 2019) skipped for both variants** -- its real analyser data
only starts Feb 2020, so pre-anchor history is empty at those anchors (same pre-established
data-scarcity class as everywhere else in this project, not new; confirmed directly via a full
traceback rather than assumed, and guarded against rather than silently producing a NaN-poisoned
metric).

**Outcome: a small, real, TabPFN-specific case for pooling, not large enough to force an
immediate switch given the deadline, but worth recording as a legitimate improvement lead.**
`BASE+species` (solo) remains the currently-documented champion (D-80, MASE=0.715 in the original
5-anchor headline table -- this notebook's own solo recompute lands at 0.7166, a small methodology-
consistent discrepancy from n-weighting/aggregation details, not a contradiction, since both solo
and pooled numbers here are computed identically). If forecasting is revisited again, TabPFN+pooled
is a concrete, cheap (no retraining, same call cost) next config to validate at full scope (more
anchors, or the downstream S-05/UQ chain) before promoting it.
